In [ ]:

from flask import Flask, request, redirect, url_for, session, flash, render_template_string
from werkzeug.security import generate_password_hash, check_password_hash
from werkzeug.utils import secure_filename
import sqlite3
import os
import threading
import time
import uuid
from functools import wraps

app = Flask(__name__)
app.secret_key = "allmarket_secret_key"

app.config["UPLOAD_FOLDER"] = "static/uploads"
app.config["MAX_CONTENT_LENGTH"] = 5 * 1024 * 1024

os.makedirs(app.config["UPLOAD_FOLDER"], exist_ok=True)

CATEGORIES = ["Bikes", "Cars", "Electronics", "Dresses"]
ALLOWED_EXTENSIONS = {"png", "jpg", "jpeg", "webp"}

STATUSES = [
    "Pending",
    "Confirmed",
    "Shipped",
    "Out for Delivery",
    "Delivered",
    "Cancelled"
]


def connect():
    db = sqlite3.connect("marketplace.db", timeout=30)
    db.row_factory = sqlite3.Row
    return db


def initialize():
    with connect() as db:
        db.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                email TEXT UNIQUE NOT NULL,
                password TEXT NOT NULL,
                role TEXT NOT NULL DEFAULT 'user'
            )
        """)

        db.execute("""
            CREATE TABLE IF NOT EXISTS products (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL,
                category TEXT NOT NULL,
                brand TEXT NOT NULL,
                price REAL NOT NULL,
                item_condition TEXT NOT NULL,
                location TEXT NOT NULL,
                description TEXT NOT NULL,
                image TEXT,
                seller_id INTEGER NOT NULL,
                status TEXT NOT NULL DEFAULT 'pending',
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY(seller_id) REFERENCES users(id)
            )
        """)

        db.execute("""
            CREATE TABLE IF NOT EXISTS wishlist (
                user_id INTEGER NOT NULL,
                product_id INTEGER NOT NULL,
                UNIQUE(user_id, product_id)
            )
        """)

        db.execute("""
            CREATE TABLE IF NOT EXISTS orders (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                buyer_id INTEGER NOT NULL,
                product_id INTEGER NOT NULL,
                buyer_name TEXT NOT NULL,
                phone TEXT NOT NULL,
                address TEXT NOT NULL,
                status TEXT NOT NULL DEFAULT 'Pending',
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)

        db.execute("""
            CREATE TABLE IF NOT EXISTS addresses (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                user_id INTEGER UNIQUE NOT NULL,
                full_name TEXT NOT NULL,
                phone TEXT NOT NULL,
                address TEXT NOT NULL,
                city TEXT NOT NULL,
                state TEXT NOT NULL,
                pincode TEXT NOT NULL
            )
        """)

        admin = db.execute(
            "SELECT id FROM users WHERE email=?",
            ("admin@marketplace.local",)
        ).fetchone()

        if not admin:
            db.execute("""
                INSERT INTO users(name,email,password,role)
                VALUES(?,?,?,?)
            """, (
                "Admin",
                "admin@marketplace.local",
                generate_password_hash("AdminChange123!"),
                "admin"
            ))
        else:
            db.execute("""
                UPDATE users SET role='admin'
                WHERE email=?
            """, ("admin@marketplace.local",))

        db.commit()


initialize()


def login_required(f):
    @wraps(f)
    def wrapper(*args, **kwargs):
        if "user_id" not in session:
            flash("Please login first.")
            return redirect(url_for("login"))
        return f(*args, **kwargs)
    return wrapper


def admin_required(f):
    @wraps(f)
    def wrapper(*args, **kwargs):
        if session.get("role") != "admin":
            flash("Admin access required.")
            return redirect(url_for("home"))
        return f(*args, **kwargs)
    return wrapper


def allowed_file(filename):
    return (
        "." in filename
        and filename.rsplit(".", 1)[1].lower() in ALLOWED_EXTENSIONS
    )


STYLE = """
* { box-sizing: border-box; }

body {
    margin: 0;
    font-family: Arial, sans-serif;
    color: white;
    background: linear-gradient(135deg,#080d24,#171044,#24105c);
    background-attachment: fixed;
}

nav {
    background: #111827;
    padding: 18px 5%;
    display: flex;
    justify-content: space-between;
    align-items: center;
    gap: 15px;
    flex-wrap: wrap;
    border-bottom: 1px solid #26344d;
}

nav a {
    color: #d5e3ff;
    text-decoration: none;
    margin: 6px 10px 6px 0;
    display: inline-block;
}

.brand {
    font-size: 25px;
    font-weight: bold;
    color: white;
}

main {
    max-width: 1150px;
    margin: 28px auto;
    padding: 0 18px;
}

.hero {
    background: linear-gradient(135deg,#172b4d,#30204d);
    padding: 38px;
    border-radius: 20px;
    margin-bottom: 30px;
}

input, select, textarea {
    width: 100%;
    padding: 13px;
    margin: 7px 0 15px;
    border: 1px solid #34415c;
    border-radius: 9px;
    background: #111a2c;
    color: white;
}

textarea {
    min-height: 100px;
    resize: vertical;
}

button, .btn {
    display: inline-block;
    background: #3978f6;
    color: white;
    border: 0;
    padding: 11px 17px;
    border-radius: 9px;
    cursor: pointer;
    font-size: 14px;
    text-decoration: none;
    margin: 4px 0;
}

button:hover, .btn:hover {
    opacity: .85;
}

.grid {
    display: grid;
    grid-template-columns: repeat(auto-fit,minmax(230px,1fr));
    gap: 20px;
}

.card {
    background: linear-gradient(145deg,#191932,#101020);
    border: 1px solid #38305c;
    border-radius: 18px;
    overflow: hidden;
}

.card img {
    width: 100%;
    height: 210px;
    object-fit: cover;
}

.info {
    padding: 17px;
}

.price {
    font-size: 23px;
    font-weight: bold;
    color: #76e0b5;
    margin: 10px 0;
}

.small {
    color: #aab7d0;
    font-size: 14px;
}

.panel {
    background: #111827;
    padding: 24px;
    border-radius: 14px;
    margin: 18px 0;
    border: 1px solid #26344d;
}

.notice {
    padding: 13px;
    background: #263b5d;
    border-radius: 9px;
    margin: 12px 0;
}

.detail-image {
    width: 100%;
    max-height: 450px;
    object-fit: contain;
}

.image-placeholder {
    height: 210px;
    display: grid;
    place-items: center;
    background: #202b43;
    font-size: 55px;
}

.like-btn {
    background: linear-gradient(135deg,#ff416c,#ff4b2b);
    width: 100%;
    margin-bottom: 10px;
}

table {
    width: 100%;
    border-collapse: collapse;
}

td, th {
    padding: 11px;
    border-bottom: 1px solid #34415c;
    text-align: left;
}

footer {
    text-align: center;
    color: #aab7d0;
    padding: 35px;
}

@media(max-width:600px) {
    nav { align-items: flex-start; }
    .hero { padding: 23px; }
}
"""


def page(content, **context):
    template = """
    <!doctype html>
    <html lang="en">
    <head>
        <meta charset="utf-8">
        <meta name="viewport" content="width=device-width,initial-scale=1">
        <title>AllMarket Shopping</title>
        <style>""" + STYLE + """</style>
    </head>
    <body>
        <nav>
            <a class="brand" href="/">🛍️ ALL MARKET</a>
            <div>
                <a href="/">Home</a>
                <a href="/?category=Bikes">Bikes</a>
                <a href="/?category=Cars">Cars</a>
                <a href="/?category=Electronics">Electronics</a>
                <a href="/?category=Dresses">Dresses</a>

                {% if session.get('user_id') %}
                    <a href="/sell">Sell Item</a>
                    <a href="/wishlist">❤️ Favorites</a>
                    <a href="/my-orders">📦 My Orders</a>
                    <a href="/my-address">🏠 My Address</a>
                    <a href="/my-listings">My Listings</a>

                    {% if session.get('role') == 'admin' %}
                        <a href="/admin">Admin</a>
                        <a href="/admin/orders">Manage Orders</a>
                    {% endif %}

                    <a href="/logout">Logout</a>
                {% else %}
                    <a href="/login">Login</a>
                    <a href="/register">Register</a>
                {% endif %}
            </div>
        </nav>

        <main>
            {% for message in get_flashed_messages() %}
                <div class="notice">{{ message }}</div>
            {% endfor %}
            """ + content + """
        </main>

        <footer>AllMarket Shopping | Buy and Sell Online</footer>
    </body>
    </html>
    """
    return render_template_string(template, **context)


CARD = """
<div class="card">
    {% if p['image'] %}
        {% if p['image'].startswith('http') %}
            <img src="{{ p['image'] }}" alt="{{ p['title'] }}">
        {% else %}
            <img src="{{ url_for('static', filename='uploads/' + p['image']) }}">
        {% endif %}
    {% else %}
        <div class="image-placeholder">🛍️</div>
    {% endif %}

    <div class="info">
        <div class="small">{{ p['category'] }} · {{ p['item_condition'] }}</div>
        <h3>{{ p['title'] }}</h3>
        <div class="price">₹{{ "%.2f"|format(p['price']) }}</div>
        <p class="small">{{ p['brand'] }} · {{ p['location'] }}</p>

        {% if session.get('user_id') %}
        <form method="post"
              action="{{ url_for('toggle_wishlist', product_id=p['id']) }}">
            <button class="like-btn" type="submit">❤️ Add / Remove Favorite</button>
        </form>
        {% endif %}

        <a class="btn" href="{{ url_for('product', product_id=p['id']) }}">
            View Details
        </a>

        <a class="btn" href="{{ url_for('buy_product', product_id=p['id']) }}">
            🛒 Buy Now
        </a>
    </div>
</div>
"""


@app.route("/")
def home():
    category = request.args.get("category", "")
    search = request.args.get("q", "").strip()
    min_price = request.args.get("min_price", "").strip()
    max_price = request.args.get("max_price", "").strip()

    sql = "SELECT * FROM products WHERE status='approved'"
    args = []

    if category in CATEGORIES:
        sql += " AND category=?"
        args.append(category)

    if search:
        sql += " AND (title LIKE ? OR brand LIKE ? OR location LIKE ?)"
        args.extend([f"%{search}%"] * 3)

    try:
        if min_price:
            sql += " AND price>=?"
            args.append(max(0, float(min_price)))
        if max_price:
            sql += " AND price<=?"
            args.append(max(0, float(max_price)))
    except ValueError:
        flash("Enter a valid price.")

    sql += " ORDER BY id DESC"

    with connect() as db:
        products = db.execute(sql, args).fetchall()

    content = """
    <section class="hero">
        <h1>Find it. Love it. Own it.</h1>
        <p>Buy and sell bikes, cars, electronics and dresses.</p>

        <form method="get">
            <input name="q" placeholder="Search products..."
                   value="{{ request.args.get('q','') }}">

            <select name="category">
                <option value="">All Categories</option>
                {% for c in categories %}
                    <option value="{{ c }}"
                    {% if request.args.get('category') == c %}selected{% endif %}>
                        {{ c }}
                    </option>
                {% endfor %}
            </select>

            <input type="number" name="min_price" min="0" placeholder="Minimum price">
            <input type="number" name="max_price" min="0" placeholder="Maximum price">
            <button>Search Products</button>
        </form>
    </section>

    <h2>Latest Products</h2>
    {% if products %}
        <div class="grid">
            {% for p in products %}
                """ + CARD + """
            {% endfor %}
        </div>
    {% else %}
        <div class="panel">No products found.</div>
    {% endif %}
    """
    return page(content, products=products, categories=CATEGORIES)


@app.route("/register", methods=["GET", "POST"])
def register():
    if request.method == "POST":
        name = request.form.get("name", "").strip()
        email = request.form.get("email", "").strip().lower()
        password = request.form.get("password", "")

        if not name or "@" not in email or len(password) < 8:
            flash("Enter valid details. Password needs 8 characters.")
        else:
            try:
                with connect() as db:
                    db.execute("""
                        INSERT INTO users(name,email,password)
                        VALUES(?,?,?)
                    """, (name, email, generate_password_hash(password)))
                    db.commit()
                flash("Registration successful. Please login.")
                return redirect(url_for("login"))
            except sqlite3.IntegrityError:
                flash("Email already registered.")

    return page("""
    <form class="panel" method="post">
        <h1>Create Account</h1>
        <input name="name" placeholder="Name" required>
        <input name="email" type="email" placeholder="Email" required>
        <input name="password" type="password" minlength="8" placeholder="Password" required>
        <button>Register</button>
    </form>
    """)


@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        email = request.form.get("email", "").strip().lower()
        password = request.form.get("password", "")

        with connect() as db:
            user = db.execute(
                "SELECT * FROM users WHERE email=?",
                (email,)
            ).fetchone()

        if user and check_password_hash(user["password"], password):
            session.clear()
            session["user_id"] = user["id"]
            session["name"] = user["name"]
            session["role"] = user["role"]

            if user["role"] == "admin":
                return redirect(url_for("admin"))
            return redirect(url_for("home"))

        flash("Incorrect email or password.")

    return page("""
    <form class="panel" method="post">
        <h1>Login</h1>
        <input type="email" name="email" placeholder="Email" required>
        <input type="password" name="password" placeholder="Password" required>
        <button>Login</button>
    </form>
    """)


@app.route("/logout")
def logout():
    session.clear()
    flash("You have logged out.")
    return redirect(url_for("home"))


@app.route("/sell", methods=["GET", "POST"])
@login_required
def sell():
    if request.method == "POST":
        title = request.form.get("title", "").strip()
        category = request.form.get("category", "")
        brand = request.form.get("brand", "").strip()
        condition = request.form.get("condition", "")
        location = request.form.get("location", "").strip()
        description = request.form.get("description", "").strip()

        try:
            price = float(request.form.get("price", ""))
        except ValueError:
            price = -1

        image = request.files.get("image")
        filename = None

        if not title or category not in CATEGORIES:
            flash("Enter valid title and category.")
        elif not brand or not location or not description:
            flash("Complete all product details.")
        elif condition not in ["New", "Used"]:
            flash("Select New or Used.")
        elif price < 0:
            flash("Enter a valid price.")
        else:
            if image and image.filename:
                if not allowed_file(image.filename):
                    flash("Use PNG, JPG, JPEG or WEBP image.")
                    return redirect(url_for("sell"))

                filename = uuid.uuid4().hex + "_" + secure_filename(image.filename)
                image.save(os.path.join(app.config["UPLOAD_FOLDER"], filename))

            with connect() as db:
                db.execute("""
                    INSERT INTO products(
                        title,category,brand,price,item_condition,
                        location,description,image,seller_id,status
                    )
                    VALUES(?,?,?,?,?,?,?,?,?,?)
                """, (
                    title, category, brand, price, condition,
                    location, description, filename,
                    session["user_id"], "pending"
                ))
                db.commit()

            flash("Product submitted for admin approval.")
            return redirect(url_for("my_listings"))

    return page("""
    <form class="panel" method="post" enctype="multipart/form-data">
        <h1>Sell Your Item</h1>
        <input name="title" placeholder="Product name" required>

        <select name="category" required>
            <option value="">Choose category</option>
            {% for c in categories %}
                <option value="{{ c }}">{{ c }}</option>
            {% endfor %}
        </select>

        <input name="brand" placeholder="Brand / Model" required>
        <input type="number" name="price" min="0" step="0.01" placeholder="Price" required>

        <select name="condition" required>
            <option>New</option>
            <option>Used</option>
        </select>

        <input name="location" placeholder="Location" required>
        <textarea name="description" placeholder="Description" required></textarea>
        <input type="file" name="image" accept=".png,.jpg,.jpeg,.webp">
        <button>Submit Listing</button>
    </form>
    """, categories=CATEGORIES)


@app.route("/product/<int:product_id>")
def product(product_id):
    with connect() as db:
        p = db.execute("""
            SELECT products.*, users.name AS seller_name,
                   users.email AS seller_email
            FROM products
            JOIN users ON products.seller_id=users.id
            WHERE products.id=? AND products.status='approved'
        """, (product_id,)).fetchone()

        saved = False
        if p and session.get("user_id"):
            saved = bool(db.execute("""
                SELECT 1 FROM wishlist
                WHERE user_id=? AND product_id=?
            """, (session["user_id"], product_id)).fetchone())

    if not p:
        return page("<div class='panel'><h1>Product not found</h1></div>"), 404

    content = """
    <div class="panel">
        {% if p['image'] %}
            {% if p['image'].startswith('http') %}
                <img class="detail-image" src="{{ p['image'] }}">
            {% else %}
                <img class="detail-image"
                     src="{{ url_for('static', filename='uploads/' + p['image']) }}">
            {% endif %}
        {% endif %}

        <h1>{{ p['title'] }}</h1>
        <div class="price">₹{{ "%.2f"|format(p['price']) }}</div>
        <p>Category: {{ p['category'] }}</p>
        <p>Brand: {{ p['brand'] }}</p>
        <p>Condition: {{ p['item_condition'] }}</p>
        <p>Location: {{ p['location'] }}</p>
        <p>{{ p['description'] }}</p>
        <p>Seller: {{ p['seller_name'] }}</p>

        {% if session.get('user_id') and session.get('user_id') != p['seller_id'] %}
            <form method="post"
                  action="{{ url_for('toggle_wishlist', product_id=p['id']) }}">
                <button>
                    {% if saved %}❤️ Remove Favorite{% else %}❤️ Add Favorite{% endif %}
                </button>
            </form>
            <a class="btn" href="{{ url_for('buy_product', product_id=p['id']) }}">Buy Now</a>
        {% elif not session.get('user_id') %}
            <a class="btn" href="/login">Login to Buy</a>
        {% else %}
            <p>This is your own listing.</p>
        {% endif %}

        <br><br>
        <a class="btn" href="/">Back to Products</a>
    </div>
    """
    return page(content, p=p, saved=saved)


@app.route("/wishlist/toggle/<int:product_id>", methods=["POST"])
@login_required
def toggle_wishlist(product_id):
    with connect() as db:
        product = db.execute("""
            SELECT id FROM products
            WHERE id=? AND status='approved'
        """, (product_id,)).fetchone()

        if not product:
            flash("Product not found.")
            return redirect(url_for("home"))

        existing = db.execute("""
            SELECT 1 FROM wishlist
            WHERE user_id=? AND product_id=?
        """, (session["user_id"], product_id)).fetchone()

        if existing:
            db.execute("""
                DELETE FROM wishlist
                WHERE user_id=? AND product_id=?
            """, (session["user_id"], product_id))
            flash("Removed from Favorites.")
        else:
            db.execute("""
                INSERT OR IGNORE INTO wishlist(user_id,product_id)
                VALUES(?,?)
            """, (session["user_id"], product_id))
            flash("Added to Favorites.")

        db.commit()

    return redirect(request.referrer or url_for("wishlist"))


@app.route("/wishlist")
@login_required
def wishlist():
    with connect() as db:
        products = db.execute("""
            SELECT p.* FROM products p
            JOIN wishlist w ON p.id=w.product_id
            WHERE w.user_id=? AND p.status='approved'
            ORDER BY p.id DESC
        """, (session["user_id"],)).fetchall()

    content = """
    <h1>❤️ My Favorites</h1>
    {% if products %}
        <div class="grid">
            {% for p in products %}
                """ + CARD + """
            {% endfor %}
        </div>
    {% else %}
        <div class="panel">Your Favorites list is empty.</div>
    {% endif %}
    """
    return page(content, products=products)


@app.route("/buy/<int:product_id>")
@login_required
def buy_product(product_id):
    with connect() as db:
        product = db.execute("""
            SELECT * FROM products
            WHERE id=? AND status='approved'
        """, (product_id,)).fetchone()

        saved = db.execute("""
            SELECT * FROM addresses WHERE user_id=?
        """, (session["user_id"],)).fetchone()

    if not product:
        flash("Product not available.")
        return redirect(url_for("home"))

    if product["seller_id"] == session["user_id"]:
        flash("You cannot buy your own product.")
        return redirect(url_for("product", product_id=product_id))

    return page("""
    <div class="panel">
        <h1>🛒 Confirm Your Order</h1>
        <h2>{{ product['title'] }}</h2>
        <div class="price">₹{{ "%.2f"|format(product['price']) }}</div>

        <form method="post"
              action="{{ url_for('confirm_purchase', product_id=product['id']) }}">

            <label>Full Name</label>
            <input name="buyer_name"
                   value="{{ saved['full_name'] if saved else session.get('name','') }}"
                   required>

            <label>Phone Number</label>
            <input name="phone"
                   value="{{ saved['phone'] if saved else '' }}"
                   required>

            <label>House Address</label>
            <textarea name="address" required>{% if saved %}{{ saved['address'] }}, {{ saved['city'] }}, {{ saved['state'] }} - {{ saved['pincode'] }}{% endif %}</textarea>

            <button type="submit">Place Order</button>
        </form>
    </div>
    """, product=product, saved=saved)


@app.route("/confirm-purchase/<int:product_id>", methods=["POST"])
@login_required
def confirm_purchase(product_id):
    buyer_name = request.form.get("buyer_name", "").strip()
    phone = request.form.get("phone", "").strip()
    address = request.form.get("address", "").strip()

    if not buyer_name or not phone or not address:
        flash("Please enter all address details.")
        return redirect(url_for("buy_product", product_id=product_id))

    with connect() as db:
        product = db.execute("""
            SELECT * FROM products
            WHERE id=? AND status='approved'
        """, (product_id,)).fetchone()

        if not product:
            flash("Product not available.")
            return redirect(url_for("home"))

        if product["seller_id"] == session["user_id"]:
            flash("You cannot buy your own product.")
            return redirect(url_for("home"))

        db.execute("""
            INSERT INTO orders(
                buyer_id,product_id,buyer_name,phone,address
            )
            VALUES(?,?,?,?,?)
        """, (
            session["user_id"],
            product_id,
            buyer_name,
            phone,
            address
        ))
        db.commit()

        order_id = db.execute("SELECT last_insert_rowid()").fetchone()[0]

    return page("""
    <div class="panel">
        <h1>🎉 Order Placed Successfully!</h1>
        <h2>{{ product_name }}</h2>
        <p>Your Order ID: #{{ order_id }}</p>
        <p>Your order status is Pending.</p>
        <a class="btn" href="{{ url_for('my_orders') }}">View My Orders</a>
        <a class="btn" href="/">Continue Shopping</a>
    </div>
    """, product_name=product["title"], order_id=order_id)


@app.route("/my-orders")
@login_required
def my_orders():
    with connect() as db:
        orders = db.execute("""
            SELECT orders.*, products.title, products.price, products.image
            FROM orders
            JOIN products ON orders.product_id=products.id
            WHERE orders.buyer_id=?
            ORDER BY orders.id DESC
        """, (session["user_id"],)).fetchall()

    return page("""
    <h1>📦 My Orders</h1>
    {% if orders %}
        <div class="grid">
            {% for o in orders %}
                <div class="card">
                    {% if o['image'] %}
                        {% if o['image'].startswith('http') %}
                            <img src="{{ o['image'] }}">
                        {% else %}
                            <img src="{{ url_for('static', filename='uploads/' + o['image']) }}">
                        {% endif %}
                    {% else %}
                        <div class="image-placeholder">🛍️</div>
                    {% endif %}
                    <div class="info">
                        <h3>{{ o['title'] }}</h3>
                        <div class="price">₹{{ "%.2f"|format(o['price']) }}</div>
                        <p>Order ID: #{{ o['id'] }}</p>
                        <p>Status: <b>{{ o['status'] }}</b></p>
                        <a class="btn" href="{{ url_for('track_order', order_id=o['id']) }}">📍 Track Order</a>
                    </div>
                </div>
            {% endfor %}
        </div>
    {% else %}
        <div class="panel">
            <h2>No Orders Yet</h2>
            <a class="btn" href="/">Start Shopping</a>
        </div>
    {% endif %}
    """, orders=orders)


@app.route("/track-order/<int:order_id>")
@login_required
def track_order(order_id):
    with connect() as db:
        order = db.execute("""
            SELECT orders.*, products.title
            FROM orders
            JOIN products ON orders.product_id=products.id
            WHERE orders.id=? AND orders.buyer_id=?
        """, (order_id, session["user_id"])).fetchone()

    if not order:
        flash("Order not found.")
        return redirect(url_for("my_orders"))

    if order["status"] == "Cancelled":
        return page("""
        <div class="panel">
            <h1>📍 Order Tracking</h1>
            <h2>{{ order['title'] }}</h2>
            <h2>❌ Order Cancelled</h2>
            <a class="btn" href="/my-orders">Back to Orders</a>
        </div>
        """, order=order)

    return page("""
    <div class="panel">
        <h1>📍 Track Your Order</h1>
        <h2>{{ order['title'] }}</h2>
        <p>Order ID: #{{ order['id'] }}</p>

        {% for s in statuses %}
            <div class="notice">
                {% if statuses.index(s) <= current_index %}
                    ✅
                {% else %}
                    ⚪
                {% endif %}
                {{ s }}
            </div>
        {% endfor %}

        <h3>Current Status: {{ order['status'] }}</h3>
        <a class="btn" href="/my-orders">Back to Orders</a>
    </div>
    """, order=order, statuses=STATUSES,
       current_index=STATUSES.index(order["status"]))


@app.route("/my-address", methods=["GET", "POST"])
@login_required
def my_address():
    with connect() as db:
        if request.method == "POST":
            name = request.form.get("full_name", "").strip()
            phone = request.form.get("phone", "").strip()
            address = request.form.get("address", "").strip()
            city = request.form.get("city", "").strip()
            state = request.form.get("state", "").strip()
            pincode = request.form.get("pincode", "").strip()

            if not all([name, phone, address, city, state, pincode]):
                flash("Please fill all address details.")
            else:
                db.execute("""
                    INSERT INTO addresses(
                        user_id,full_name,phone,address,city,state,pincode
                    )
                    VALUES(?,?,?,?,?,?,?)
                    ON CONFLICT(user_id) DO UPDATE SET
                        full_name=excluded.full_name,
                        phone=excluded.phone,
                        address=excluded.address,
                        city=excluded.city,
                        state=excluded.state,
                        pincode=excluded.pincode
                """, (
                    session["user_id"], name, phone,
                    address, city, state, pincode
                ))
                db.commit()
                flash("Address saved successfully!")
                return redirect(url_for("my_address"))

        saved = db.execute("""
            SELECT * FROM addresses WHERE user_id=?
        """, (session["user_id"],)).fetchone()

    return page("""
    <form class="panel" method="post">
        <h1>🏠 My Delivery Address</h1>

        <label>Full Name</label>
        <input name="full_name"
               value="{{ saved['full_name'] if saved else '' }}" required>

        <label>Phone Number</label>
        <input name="phone"
               value="{{ saved['phone'] if saved else '' }}" required>

        <label>House Address</label>
        <textarea name="address" required>{{ saved['address'] if saved else '' }}</textarea>

        <label>City</label>
        <input name="city" value="{{ saved['city'] if saved else '' }}" required>

        <label>State</label>
        <input name="state" value="{{ saved['state'] if saved else '' }}" required>

        <label>Pincode</label>
        <input name="pincode" value="{{ saved['pincode'] if saved else '' }}" required>

        <button type="submit">💾 Save Address</button>
    </form>
    """, saved=saved)


@app.route("/my-listings")
@login_required
def my_listings():
    with connect() as db:
        products = db.execute("""
            SELECT * FROM products
            WHERE seller_id=?
            ORDER BY id DESC
        """, (session["user_id"],)).fetchall()

    return page("""
    <h1>My Listings</h1>
    {% if products %}
        <div class="grid">
            {% for p in products %}
                <div class="card">
                    {% if p['image'] %}
                        {% if p['image'].startswith('http') %}
                            <img src="{{ p['image'] }}">
                        {% else %}
                            <img src="{{ url_for('static', filename='uploads/' + p['image']) }}">
                        {% endif %}
                    {% endif %}
                    <div class="info">
                        <h3>{{ p['title'] }}</h3>
                        <div class="price">₹{{ "%.2f"|format(p['price']) }}</div>
                        <p>Status: {{ p['status'] }}</p>
                        <form method="post" action="{{ url_for('delete_listing', product_id=p['id']) }}">
                            <button>Remove Listing</button>
                        </form>
                    </div>
                </div>
            {% endfor %}
        </div>
    {% else %}
        <div class="panel">No listings yet.</div>
    {% endif %}
    """, products=products)


@app.route("/delete-listing/<int:product_id>", methods=["POST"])
@login_required
def delete_listing(product_id):
    with connect() as db:
        db.execute("""
            UPDATE products SET status='removed'
            WHERE id=? AND seller_id=?
        """, (product_id, session["user_id"]))
        db.commit()

    flash("Listing removed.")
    return redirect(url_for("my_listings"))


@app.route("/admin")
@login_required
@admin_required
def admin():
    with connect() as db:
        products = db.execute("""
            SELECT p.*, u.name AS seller_name
            FROM products p
            JOIN users u ON p.seller_id=u.id
            ORDER BY p.id DESC
        """).fetchall()

        users = db.execute("""
            SELECT id,name,email,role FROM users ORDER BY id DESC
        """).fetchall()

    return page("""
    <h1>Admin Dashboard</h1>
    <h2>Manage Listings</h2>
    <div class="panel">
        <table>
            <tr>
                <th>Product</th><th>Seller</th><th>Price</th>
                <th>Status</th><th>Action</th>
            </tr>
            {% for p in products %}
            <tr>
                <td>{{ p['title'] }}</td>
                <td>{{ p['seller_name'] }}</td>
                <td>₹{{ p['price'] }}</td>
                <td>{{ p['status'] }}</td>
                <td>
                    {% if p['status'] == 'pending' %}
                    <form method="post" action="{{ url_for('moderate', product_id=p['id'], action='approve') }}">
                        <button>Approve</button>
                    </form>
                    {% endif %}
                    <form method="post" action="{{ url_for('moderate', product_id=p['id'], action='remove') }}">
                        <button>Remove</button>
                    </form>
                </td>
            </tr>
            {% endfor %}
        </table>
    </div>

    <h2>Registered Users</h2>
    <div class="panel">
        <table>
            <tr><th>Name</th><th>Email</th><th>Role</th></tr>
            {% for u in users %}
            <tr>
                <td>{{ u['name'] }}</td>
                <td>{{ u['email'] }}</td>
                <td>{{ u['role'] }}</td>
            </tr>
            {% endfor %}
        </table>
    </div>
    """, products=products, users=users)


@app.route("/admin/moderate/<int:product_id>/<action>", methods=["POST"])
@login_required
@admin_required
def moderate(product_id, action):
    status = {
        "approve": "approved",
        "remove": "removed"
    }.get(action)

    if status:
        with connect() as db:
            db.execute(
                "UPDATE products SET status=? WHERE id=?",
                (status, product_id)
            )
            db.commit()
        flash("Listing updated.")

    return redirect(url_for("admin"))


@app.route("/admin/orders")
@login_required
@admin_required
def admin_orders():
    with connect() as db:
        orders = db.execute("""
            SELECT orders.*, products.title
            FROM orders
            JOIN products ON orders.product_id=products.id
            ORDER BY orders.id DESC
        """).fetchall()

    return page("""
    <div class="panel">
        <h1>📦 Manage Orders</h1>
        {% for o in orders %}
            <div class="notice">
                <h3>{{ o['title'] }}</h3>
                <p>Order ID: #{{ o['id'] }}</p>
                <p>Customer: {{ o['buyer_name'] }}</p>
                <p>Phone: {{ o['phone'] }}</p>
                <p>Address: {{ o['address'] }}</p>
                <p>Status: {{ o['status'] }}</p>

                <form method="post"
                      action="{{ url_for('update_order_status', order_id=o['id']) }}">
                    <select name="status">
                        {% for s in statuses %}
                            <option value="{{ s }}"
                            {% if o['status'] == s %}selected{% endif %}>
                                {{ s }}
                            </option>
                        {% endfor %}
                    </select>
                    <button>Update Status</button>
                </form>
            </div>
        {% else %}
            <p>No orders available.</p>
        {% endfor %}
    </div>
    """, orders=orders, statuses=STATUSES)


@app.route("/admin/update-order/<int:order_id>", methods=["POST"])
@login_required
@admin_required
def update_order_status(order_id):
    status = request.form.get("status")

    if status not in STATUSES:
        flash("Invalid order status.")
        return redirect(url_for("admin_orders"))

    with connect() as db:
        db.execute("""
            UPDATE orders SET status=? WHERE id=?
        """, (status, order_id))
        db.commit()

    flash("Order status updated.")
    return redirect(url_for("admin_orders"))


@app.errorhandler(413)
def too_large(error):
    return page("""
    <div class="panel">
        <h1>Image too large</h1>
        <p>Please upload an image smaller than 5 MB.</p>
    </div>
    """), 413


def add_sample_products():
    with connect() as db:
        admin = db.execute("""
            SELECT id FROM users WHERE email=?
        """, ("admin@marketplace.local",)).fetchone()

        if not admin:
            return

        existing = db.execute("""
            SELECT COUNT(*) FROM products WHERE seller_id=?
        """, (admin["id"],)).fetchone()[0]

        if existing > 0:
            return

        seller_id = admin["id"]

        products = [
            ("Royal Enfield Classic 350", "Bikes", "Royal Enfield", 185000, "Used", "Chennai", "Well maintained bike", "https://images.unsplash.com/photo-1558981806-ec527fa84c39?auto=format&fit=crop&w=800&q=80", seller_id, "approved"),
            ("Royal Enfield Hunter 350", "Bikes", "Royal Enfield", 165000, "Used", "Coimbatore", "Good condition bike", "https://images.unsplash.com/photo-1558980394-0cbbf08f4b7e?auto=format&fit=crop&w=800&q=80", seller_id, "approved"),
            ("Yamaha R15 V4", "Bikes", "Yamaha", 195000, "Used", "Coimbatore", "Excellent condition", "https://images.unsplash.com/photo-1558980664-10ea95b8a2b9?auto=format&fit=crop&w=800&q=80", seller_id, "approved"),
            ("Honda Activa 6G", "Bikes", "Honda", 85000, "Used", "Madurai", "Family use scooter", "https://images.unsplash.com/photo-1591637333184-19aa84b3e01f?auto=format&fit=crop&w=800&q=80", seller_id, "approved"),
            ("Maruti Suzuki Swift", "Cars", "Maruti Suzuki", 650000, "Used", "Chennai", "Well maintained car", "https://images.unsplash.com/photo-1492144534655-ae79c964c9d7?auto=format&fit=crop&w=800&q=80", seller_id, "approved"),
            ("Hyundai Creta", "Cars", "Hyundai", 1250000, "Used", "Coimbatore", "Excellent condition", "https://images.unsplash.com/photo-1549317661-bd32c8ce0db2?auto=format&fit=crop&w=800&q=80", seller_id, "approved"),
            ("iPhone 15", "Electronics", "Apple", 65000, "Used", "Chennai", "Excellent condition phone", "https://images.unsplash.com/photo-1592750475338-74b7b21085ab?auto=format&fit=crop&w=800&q=80", seller_id, "approved"),
            ("Samsung Galaxy S24", "Electronics", "Samsung", 55000, "Used", "Coimbatore", "Good condition phone", "https://images.unsplash.com/photo-1610945415295-d9bbf067e59c?auto=format&fit=crop&w=800&q=80", seller_id, "approved"),
            ("Dell Inspiron Laptop", "Electronics", "Dell", 45000, "Used", "Madurai", "Laptop for students", "https://images.unsplash.com/photo-1593642532400-2682810df593?auto=format&fit=crop&w=800&q=80", seller_id, "approved")
        ]

        db.executemany("""
            INSERT INTO products(
                title,category,brand,price,item_condition,
                location,description,image,seller_id,status
            )
            VALUES(?,?,?,?,?,?,?,?,?,?)
        """, products)

        db.commit()


add_sample_products()


def run_app():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )


if __name__ == "__main__":
    server_thread = threading.Thread(target=run_app, daemon=True)
    server_thread.start()
    time.sleep(3)

    try:
        from google.colab.output import eval_js
        website_url = eval_js("google.colab.kernel.proxyPort(5000)")
        print("Your AllMarket Website:")
        print(website_url)
    except Exception:
        print("Flask started successfully.")
        print("Open: http://127.0.0.1:5000")
